In [ ]:
%pip install pandas 

In [ ]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

In [ ]:
dfgenome_scores = pd.read_csv(r"C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\genome-scores.csv")

dfgenome_tags = pd.read_csv(r"C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\genome-tags.csv")

dflink = pd.read_csv(r"C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\links.csv")
dfmovie = pd.read_csv(r"C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\movies.csv")
dfratings = pd.read_csv(r"C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\ratings.csv")

dftags = pd.read_csv(r"C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\tags.csv")


In [ ]:
dftags.head()

In [ ]:
dfgenome_scores.head()

In [ ]:
dfratings.head()

In [ ]:
dfmovie.head()

In [ ]:
dflink.head()

In [ ]:
dfgenome_scores.head()

In [ ]:
dfgenome_tags.head()

In [ ]:
dftags.shape
dfgenome_tags.shape
dfgenome_scores.shape
dflink.shape
dfmovie.shape
dfratings.shape


In [ ]:
common_columns = ['userId', 'movieId', 'timestamp']
merged_dataset = pd.merge(dftags, dfratings, on=common_columns, how='right')


In [ ]:
merged_dataset.head()

In [ ]:
merged_dataset.shape

In [ ]:
common_column = ['movieId']
dfmerged = pd.merge(merged_dataset,dflink, on=common_column, how='right')

In [ ]:
dfmerged.shape

In [ ]:
dfmerged.head()

In [ ]:
dfmerged = dfmerged.drop(columns=['tag'])

In [ ]:
dfmerged.head()

In [ ]:
common_col = ["movieId"]
dfmerged = pd.merge(dfmovie, dfmerged, on=common_col, how="right")

In [ ]:
dfmerged.shape

In [ ]:
dfmerged.head()

In [ ]:
dfmerged["imdbId"].nunique()

In [ ]:
dfmerged[dfmerged.duplicated()].sum()

In [ ]:
dfmerged.value_counts().sum()

In [ ]:
dflink.head()

In [ ]:
dfmerged.columns

In [ ]:
dfgenome_scores.head()

In [ ]:


# Load the datasets with actual file paths
dfgenome_scores = pd.read_csv(r'C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\genome-scores.csv')
dfmovies = dfmerged 

# Define the common column for merging
common_coll = ['movieId']

# Step 1: Select only necessary columns to reduce memory usage
dfgenome_scores_small = dfgenome_scores[['movieId', 'tagId', 'relevance']]
dfmovies_small = dfmovies[['movieId', 'title', 'genres', 'userId', 'rating']]

# Step 2: Drop duplicates in the common column to avoid excessive rows from duplicate merges
dfgenome_scores_small = dfgenome_scores_small.drop_duplicates(subset=common_coll)
dfmovies_small = dfmovies_small.drop_duplicates(subset=common_coll)

# Step 3: Optimize data types to reduce memory usage
# Downcast numerical columns (if applicable)
for col in dfgenome_scores_small.select_dtypes(include=['float64', 'int64']).columns:
    dfgenome_scores_small[col] = pd.to_numeric(dfgenome_scores_small[col], downcast='float')
for col in dfmovies_small.select_dtypes(include=['float64', 'int64']).columns:
    dfmovies_small[col] = pd.to_numeric(dfmovies_small[col], downcast='float')

# Convert object columns to categories if they have few unique values
for col in dfgenome_scores_small.select_dtypes(include=['object']).columns:
    if dfgenome_scores_small[col].nunique() / len(dfgenome_scores_small[col]) < 0.5:
        dfgenome_scores_small[col] = dfgenome_scores_small[col].astype('category')
for col in dfmovies_small.select_dtypes(include=['object']).columns:
    if dfmovies_small[col].nunique() / len(dfmovies_small[col]) < 0.5:
        dfmovies_small[col] = dfmovies_small[col].astype('category')

# Step 4: Perform a chunked merge to handle large data efficiently
chunk_size = 100000  # Adjust chunk size based on available memory
merged_chunks = []

for chunk in pd.read_csv(r'C:\Users\SOOQ ELASER\movie_recomendation_collaborative_filtering\ml-20m\ml-20m\genome-scores.csv', chunksize=chunk_size):
    chunk = chunk[['movieId', 'tagId', 'relevance']] 
    merged_chunk = pd.merge(dfmovies_small, chunk, on=common_coll, how='inner')
    merged_chunks.append(merged_chunk)

# Combine all chunked merges into a final DataFrame
dfmerged_final = pd.concat(merged_chunks, ignore_index=True)



In [ ]:
dfmerged_final.head()

In [ ]:
dfmerged_final.shape

In [ ]:
dfmerged_final[dfmerged_final.duplicated()]

In [ ]:
dfmerged_final.isnull().sum()





In [ ]:
dfmerged_final = dfmerged_final.dropna()

In [ ]:
dfmerged_final.isnull().sum()

In [ ]:
dfmerged_final.shape

In [ ]:
dfgenome_tags.head()

In [ ]:
dfgenome_tags['tag'].nunique()

In [ ]:
dfgenome_tags.tail()

In [ ]:
dfgenome_tags.shape

## Rating Distribution Analysis

### Distribution of Ratings

The following histogram displays the frequency of ratings given by users in our movie recommendation dataset. Each bin represents a range of rating scores, allowing us to see how many ratings fall into each category.

- **Histogram Interpretation:**
  - The x-axis represents the rating scores, while the y-axis shows the number of ratings for each score.
  - Observing the shape of the histogram provides insights into user behavior; for instance, if most ratings cluster around the higher end (e.g., 4 and 5 stars), this could indicate a tendency for users to favor movies more positively.


In [ ]:
# Plotting a histogram
plt.figure(figsize=(10, 5))
sns.histplot(dfmerged_final['rating'], bins=10, kde=False)
plt.title('Distribution of Ratings')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.show()


### KDE Plot of Ratings

The KDE (Kernel Density Estimate) plot offers a smoothed representation of the rating distribution, providing a clearer view of the density of ratings across the scale.

- **KDE Plot Interpretation:**
  - Similar to the histogram, the x-axis represents the rating scores, while the y-axis indicates the density of ratings.
  - This plot allows us to identify subtle trends, such as whether users disproportionately favor certain ratings (like 4.5 or 3.0).
  - The smooth curve helps to visualize potential peaks in ratings that might not be as evident in the histogram.


In [ ]:
# Plotting a KDE plot
plt.figure(figsize=(10, 5))
sns.kdeplot(dfmerged_final['rating'], fill=True)
plt.title('KDE Plot of Ratings')
plt.xlabel('Rating')
plt.ylabel('Density')
plt.show()

### Correlation Matrix Analysis

In this section, we calculate and visualize the correlation matrix of the numeric features in the merged movie dataset. The correlation matrix helps us understand the relationships between different numeric variables, indicating how closely related they are to one another.

#### Code Explanation

1. **Import Libraries**: We import the necessary libraries for data manipulation and visualization:
   - `pandas` for data handling.
   - `seaborn` and `matplotlib.pyplot` for plotting.

2. **Selecting Numeric Columns**: We filter the DataFrame `dfmerged_final` to retain only numeric columns using `select_dtypes(include=['float64', 'int64'])`. This step is crucial as correlation calculations require numeric data.

3. **Calculating Correlation Matrix**: The `corr()` method computes the correlation coefficients between the numeric columns. The output is a square matrix where each cell shows the correlation between the respective features.

4. **Visualization**: We use Seaborn's `heatmap` function to create a visual representation of the correlation matrix. The `annot=True` parameter displays the correlation coefficients in the heatmap cells, and `cmap='coolwarm'` sets the color palette for better visualization.


In [ ]:
# Select only numeric columns for correlation analysis
numeric_df = dfmerged_final.select_dtypes(include=['float64', 'int64'])

# Calculate the correlation matrix
correlation_matrix = numeric_df.corr()

# Set up the matplotlib figure
plt.figure(figsize=(10, 8))

# Draw the heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={"shrink": .8})

# Title and labels
print(correlation_matrix)
plt.title('Correlation Matrix Heatmap', fontsize=16)
plt.show()

In [ ]:
dfmerged_final.head()

In [ ]:
dfmerged_final["title"].nunique()

In [ ]:


# Assuming dfmerged_final is your DataFrame
# Select only numeric columns
numeric_cols = dfmerged_final.select_dtypes(include=['float64', 'int64']).columns

# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the numeric columns
dfmerged_final[numeric_cols] = scaler.fit_transform(dfmerged_final[numeric_cols])

dfmerged_final.head()


In [ ]:
dfmerged_final['genres'].nunique()

In [ ]:
# List of columns to encode
columns_to_encode = ['title', 'genres']

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Apply Label Encoding to each specified column
for col in columns_to_encode:
    dfmerged_final[f'{col}_encoded'] = label_encoder.fit_transform(dfmerged_final[col])

# Display the DataFrame with encoded columns
dfmerged_final.head()

In [ ]:
# Drop the specified columns
dfmerged_final = dfmerged_final.drop(columns=['title', 'genres'])

In [ ]:
dfmerged_final.head()

In [ ]:
dfmerged_final.shape

In [ ]:
dfmerged_final.isnull().sum()